In [1]:
import psycopg as pg
import numpy as np
import pandas as pd
import os
import mlflow
import joblib
from collections import defaultdict

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from category_encoders import CatBoostEncoder
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedKFold
from catboost import CatBoostClassifier

import optuna
from optuna.integration.mlflow import MLflowCallback

In [2]:
from sklearn.metrics import roc_auc_score, confusion_matrix, f1_score, precision_score, recall_score, log_loss

In [3]:
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://storage.yandexcloud.net" #endpoint бакета от YandexCloud
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("AWS_ACCESS_KEY_ID") # получаем id ключа бакета, к которому подключён MLFlow, из .env
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("AWS_SECRET_ACCESS_KEY") # получаем ключ бакета, к которому подключён MLFlow, из .env

TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")

pd.options.display.max_columns = 100
pd.options.display.max_rows = 64

In [4]:
connection = {"sslmode": "require", "target_session_attrs": "read-write"}
postgres_credentials = {
    "host": os.getenv("DB_DESTINATION_HOST"),
    "port": os.getenv("DB_DESTINATION_PORT"),
    "dbname": os.getenv("DB_DESTINATION_NAME"),
    "user": os.getenv("DB_DESTINATION_USER"),
    "password": os.getenv("DB_DESTINATION_PASSWORD"),
}
connection.update(postgres_credentials)

TABLE_NAME = "users_churn"
with pg.connect(**connection) as conn:
    with conn.cursor() as cur:
        cur.execute(f"SELECT * FROM {TABLE_NAME}")
        data = cur.fetchall()
        columns = [col[0] for col in cur.description]

df = pd.DataFrame(data, columns=columns)

In [5]:
EXPERIMENT_NAME = "churn_vds"
REGISTRY_MODEL_NAME = "churn_model_vds"

experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if not experiment:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
else:
    experiment_id = experiment.experiment_id

In [6]:
features = ["monthly_charges", "total_charges", "senior_citizen"]
target = "target"

split_column = "begin_date"
stratify_column = "begin_date"
test_size = 0.2

df = df.sort_values(by=[split_column])

X_train, X_test, y_train, y_test = train_test_split(
    df[features],
    df['target'],
    test_size=test_size,
    shuffle=False,
)

In [7]:
def objective(trial: optuna.Trial) -> float:
    param = {
       "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.1, log=True),
       "depth": trial.suggest_int("depth", 1, 12),
       "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 0.1, 5),
       "random_strength": trial.suggest_float("random_strength", 0.1, 5),
       "loss_function": "Logloss",
       "task_type": "CPU",
       "random_seed": 0,
       "iterations": 300,
       "verbose": False,
     }
    model = CatBoostClassifier(**param)

    skf = StratifiedKFold(n_splits=2)

    metrics = defaultdict(list)
    for i, (train_index, val_index) in enumerate(skf.split(X_train, y_train)):
        train_x = X_train.iloc[train_index]
        train_y = y_train.iloc[train_index]
        val_x = X_train.iloc[val_index]
        val_y = y_train.iloc[val_index]
        
        model.fit(train_x, train_y)
        
        prediction = model.predict(val_x)
        probas = model.predict_proba(val_x)[:,1]

        _, err1, _, err2 = confusion_matrix(val_y, prediction, normalize='all').ravel()
        auc = roc_auc_score(val_y, probas)
        precision = precision_score(val_y, prediction)
        recall = recall_score(val_y, prediction)
        f1 = f1_score(val_y, prediction)
        logloss = log_loss(val_y, prediction)
        
        metrics["err1"].append(err1)
        metrics["err2"].append(err2)
        metrics["auc"].append(auc)
        metrics["precision"].append(precision)
        metrics["recall"].append(recall)
        metrics["f1"].append(f1)
        metrics["logloss"].append(logloss)


    # ваш код здесь #
    err_1 = np.median(metrics['err1'])
    err_2 = np.median(metrics['err2'])
    auc = np.median(metrics['auc'])
    precision = np.median(metrics['precision'])
    recall = np.median(metrics['recall'])
    f1 = np.median(metrics['f1'])
    logloss = np.median(metrics['logloss'])

    return auc

In [15]:
RUN_NAME = "model_bayesian_search"
with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id

    mlflc = MLflowCallback(
        tracking_uri=f'http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}',
        metric_name="AUC",
        create_experiment=False,
        mlflow_kwargs={'experiment_id': experiment_id, 'parent_run_id': run_id, "nested": True}
    )

    STUDY_DB_NAME = "sqlite:///local.study.db"
    STUDY_NAME = "churn_model_23"

    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(), study_name=STUDY_NAME, storage=STUDY_DB_NAME, load_if_exists=True)
    study.optimize(objective, n_trials=10, callbacks=[mlflc])
    best_params = study.best_params
    best_model = CatBoostClassifier(**best_params).fit(X_train, y_train)

    cv_info = mlflow.catboost.log_model( 
            cb_model=best_model,
            artifact_path='cv',
            await_registration_for=60,)
    model_info = mlflow.catboost.log_model( 
            cb_model=best_model,
            artifact_path='models',
            registered_model_name=REGISTRY_MODEL_NAME,
            await_registration_for=60,)

/tmp/ipykernel_4308/1537943742.py:5: ExperimentalWarning: MLflowCallback is experimental (supported from v1.4.0). The interface can change in the future.
  mlflc = MLflowCallback(
[I 2026-04-23 21:01:14,174] A new study created in RDB with name: churn_model_23
/home/mle-user/mle_projects/mle-mlflow/.venv_mlflow/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
[I 2026-04-23 21:01:15,102] Trial 0 finished with value: 0.8112274744914392 and parameters: {'learning_rate': 0.08135522949222694, 'depth': 4, 'l2_leaf_reg': 4.226287390086289, 'random_strength': 2.0921279725031074}. Best is trial 0 with value: 0.8112274744914392.
/home/mle-user/mle_projects/mle-mlflow/.venv_mlflow/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1497: Un

0:	learn: 0.6616925	total: 1.29ms	remaining: 1.29s
1:	learn: 0.6227568	total: 2.99ms	remaining: 1.49s
2:	learn: 0.5915468	total: 4.42ms	remaining: 1.47s
3:	learn: 0.5638185	total: 5.97ms	remaining: 1.49s
4:	learn: 0.5367480	total: 7.55ms	remaining: 1.5s
5:	learn: 0.5258702	total: 8.83ms	remaining: 1.46s
6:	learn: 0.5100189	total: 10.4ms	remaining: 1.48s
7:	learn: 0.4968025	total: 11.9ms	remaining: 1.47s
8:	learn: 0.4865137	total: 13.6ms	remaining: 1.5s
9:	learn: 0.4780819	total: 15.1ms	remaining: 1.5s
10:	learn: 0.4691510	total: 16.6ms	remaining: 1.49s
11:	learn: 0.4610253	total: 18ms	remaining: 1.49s
12:	learn: 0.4537140	total: 19.5ms	remaining: 1.48s
13:	learn: 0.4471455	total: 21ms	remaining: 1.48s
14:	learn: 0.4396145	total: 22.8ms	remaining: 1.5s
15:	learn: 0.4361548	total: 24.8ms	remaining: 1.53s
16:	learn: 0.4347016	total: 26.4ms	remaining: 1.53s
17:	learn: 0.4302781	total: 27.9ms	remaining: 1.52s
18:	learn: 0.4273824	total: 29.3ms	remaining: 1.51s
19:	learn: 0.4246490	total: 30

Registered model 'churn_model_vds' already exists. Creating a new version of this model...
2026/04/23 21:01:42 INFO mlflow.tracking._model_registry.client: Waiting up to 60 seconds for model version to finish creation. Model name: churn_model_vds, version 11
Created version '11' of model 'churn_model_vds'.


In [16]:
print(run_id)

9914cf5ee0394216a6f8fa761521918d
